## Lab 2 - Part B : Inverse Kinematics

### Overview & Variables

Part B  ให้เอาคำตอบ IK ที่แก้ **ด้วยมือใน Part A** มาตรวจด้วย Robotics Toolbox แล้ว cross-check ด้วย Forward Kinematics (FK)

| Variable | What is it? | From |
|--------|---------|-------------|
| `M` | ตำแหน่งเป้าหมายใน **base frame** ของหุ่น (= `Pobj`) หน่วยเมตร `[x, y, z]` | Part A ข้อ 1: แปลง `M` จาก camera frame มาเป็น base frame ด้วย inverse ของ `T` ที่ได้รับ |
| `q_goal` | มุม/ระยะข้อต่อที่แก้ IK **ด้วยมือ** (เลือกมา 1 branch) | Part A ข้อ 4 |
| `branches` | คำตอบ IK **ทุก branch** | Part A ข้อ 4 |

**หน่วยของ `q`** ขึ้นกับชนิดข้อต่อของหุ่นที่ออกแบบใน LAB1: revolute → เรเดียน `[rad]`, prismatic → เมตร `[m]`
(หุ่นใน template นี้เป็นแบบ **RRP** → `q1, q2` = rad, `q3` = m)

**เกณฑ์ผ่าน** error ในส่วนที่ 2 และ 4 ต้องไม่เกิน 0.03 m


### 0.1 Check your URDF  (run this first)

เช็คไฟล์ `my_robot/robot.urdf` ก่อนโหลดหุ่น ถ้าไฟล์มีปัญหา ค่า `q` ที่คำนวณมือถูกก็จะไม่ตรงใน toolbox *(nothing to edit)*

| เช็คอะไร | ถ้าไม่ผ่านจะเกิดอะไร |
|---|---|
| ไฟล์เป็น ASCII ล้วน | โหลด URDF บน Windows ไม่ได้ (`UnicodeDecodeError`) |
| มี joint ที่ขยับได้ 3 ตัวพอดี | ใส่ `q` 3 ค่าไม่ได้ |
| แกนของทุก joint เป็น x, y หรือ z พอดี | toolbox อ่านหุ่นผิด ปลายมือไม่ตรงกับที่คำนวณ |
| joint มี limit และ lower < upper | `ikine_LM` ใช้ joint limit ไม่ได้ |
| path ของ mesh ถูกและมีไฟล์จริง | หุ่นไม่แสดงใน Swift |

ต้องได้ `URDF check: OK` ก่อนทำข้อถัดไป ถ้าไม่ผ่านให้แก้ `robot.urdf` ตามรายการที่พิมพ์ออกมาแล้วรันเซลล์นี้ใหม่
และดูตาราง joint ที่พิมพ์ออกมาว่าชนิด แกน และ limit ตรงกับหุ่นที่ออกแบบใน LAB1

In [ ]:
# URDF check - run this before Setup (nothing to edit)
from pathlib import Path
import xml.etree.ElementTree as ET

ROBOT_DIR = Path('my_robot')
raw = (ROBOT_DIR / 'robot.urdf').read_bytes()
problems = []

# (1) plain ASCII only - other characters (e.g. Thai comments) stop the loader on Windows
bad_lines = [i + 1 for i, line in enumerate(raw.split(b'\n')) if max(line, default=0) > 127]
if bad_lines:
    problems.append(f'non-ASCII characters on line {bad_lines[:5]} - delete them (e.g. Thai comments)')

urdf = ET.fromstring(raw)
moving = [j for j in urdf.findall('joint') if j.get('type') != 'fixed']

# (2) exactly 3 moving joints; the end-effector joint must be fixed
kinds = ''.join('P' if j.get('type') == 'prismatic' else 'R' for j in moving)
if len(moving) != 3:
    problems.append(f'{len(moving)} moving joints ({kinds}) - need exactly 3; the end-effector joint must be type="fixed"')

print('joint                | type       | axis       | limit')
for j in moving:
    name, kind, limit = j.get('name'), j.get('type'), j.find('limit')
    axis = j.find('axis').get('xyz') if j.find('axis') is not None else '1 0 0'
    lo, hi = (limit.get('lower'), limit.get('upper')) if limit is not None else (None, None)
    print(f'{name:<20} | {kind:<10} | {axis:<10} | {lo} .. {hi}')
    # (3) the axis must be exactly x, y or z - an axis like "0.001 0 0.9999" is read wrongly by the toolbox
    if sorted(abs(float(v)) for v in axis.split()) != [0.0, 0.0, 1.0]:
        problems.append(f'joint "{name}": axis "{axis}" is not exactly x, y or z - round it (e.g. "0 0 1")')
    # (4) revolute / prismatic joints need lower < upper
    if kind != 'continuous' and (lo is None or hi is None or float(lo) >= float(hi)):
        problems.append(f'joint "{name}": needs <limit lower="..." upper="..."/> with lower < upper')

# (5) every mesh must be package://my_robot/meshes/<file> and the file must exist
for path in sorted({mesh.get('filename', '') for mesh in urdf.iter('mesh')}):
    if not path.startswith('package://my_robot/meshes/'):
        problems.append(f'mesh path "{path}" must start with package://my_robot/meshes/')
    elif not (ROBOT_DIR / 'meshes' / path.split('/')[-1]).is_file():
        problems.append(f'mesh file not found: my_robot/meshes/{path.split("/")[-1]}')

print()
if problems:
    print('URDF check: FIX THESE FIRST')
    for p in problems:
        print('  [x]', p)
else:
    print('URDF check: OK  (' + kinds + ', 3 moving joints)')

### 0.2 Setup

In [ ]:
import numpy as np
from pathlib import Path
from roboticstoolbox import Robot
from spatialmath import SE3
import spatialgeometry as sg
import swift

HERE = Path.cwd().as_posix()
links, name, _, _ = Robot.URDF_read('my_robot/robot.urdf', tld=HERE)
robot = Robot(links, name=name)
print('loaded', name, robot.n, 'joints')

env = swift.Swift()
env.launch(realtime=True, browser="notebook")
env.add(robot)
robot.q = [0.0, 0.0, 0.0]   # home configuration
env.step()

In [ ]:
# Allowable syntax (reference only - do not run as-is):
# robot.fkine(q)          # forward kinematics -> transform T ; tip position is T.t
# np.linalg.norm(a - b)   # distance between two points
# SE3(*M)                 # make a target pose from a point M = [x, y, z]
# robot.ikine_LM(SE3(*M), q0=..., mask=[1,1,1,0,0,0], joint_limits=True)   # numerical IK -> sol
# sol.q                   # the joint solution it found
# sol.success             # did it converge (True/False)

### 1. target M (from Part A item 1) and q (from Part A item 4)

In [ ]:
M = np.array([0.0, 0.0, 0.0])       # edit here
q_goal = np.array([0.0, 0.0, 0.0])  # edit here

if np.allclose(M, 0) or np.allclose(q_goal, 0):
    print('fill in M and q_goal first, then run again')
else:
    marker = sg.Sphere(radius=0.03, color=(1.0, 0.1, 0.1, 0.6))
    marker.T = SE3(*M)
    env.add(marker)
    env.step()
    print('target shown at', np.round(M, 4))

In [ ]:
# move the robot to q_goal  (tip should reach the ball)
robot.q = q_goal
env.step()

### 2. cross-check with FK

In [ ]:
q = q_goal

tip = None    # edit here
dist = None   # edit here

if dist is None:
    print('fill in tip and dist above')
else:
    print('tip:', np.round(tip, 4), '| M:', np.round(M, 4), '| err:', round(float(dist), 4))

### 3. let the toolbox solve IK (ikine), compare with q_goal

In [ ]:
sol = None    # edit here  (see hint above)

if sol is None:
    print('fill in the ikine_LM call')
else:
    print('toolbox q:', np.round(sol.q, 4), '| ok', sol.success)
    print('hand q   :', np.round(q_goal, 4))

### 4. check every branch

In [ ]:
branches = [
    np.array([0.0, 0.0, 0.0]),    # edit here : branch 1
    # np.array([0.0, 0.0, 0.0]),  # branch 2 if any
]

for i, qb in enumerate(branches):
    err = np.linalg.norm(robot.fkine(qb).t - M)
    print('branch', i + 1, np.round(qb, 4), '-> err', round(float(err), 4))